In [2]:
import pandas as pd
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# Load cleaned dataset
file_path = r"C:\Users\Haneefa Ghazi\netflix_titles_cleaned.csv"
data = pd.read_csv(file_path)

# Convert 'date_added' to datetime
data['date_added'] = pd.to_datetime(data['date_added'])

# Extract Year-Month for Trend Analysis
data['YearMonth'] = data['date_added'].dt.to_period('M')

# 1️⃣ Interactive Bar Chart: Top 10 Most Common Genres
genre_counts = data['listed_in'].str.split(', ').explode().value_counts().head(10)
fig_genre = px.bar(
    genre_counts, 
    x=genre_counts.index, 
    y=genre_counts.values,
    labels={'x': 'Genre', 'y': 'Count'},
    title='Top 10 Most Common Netflix Genres',
    color=genre_counts.index
)

# 2️⃣ Interactive Time-Series Plot: Number of Titles Added Over Time
trend = data.groupby('YearMonth').size()
fig_trend = px.line(
    x=trend.index.astype(str),
    y=trend.values,
    labels={'x': 'Year-Month', 'y': 'Number of Titles'},
    title='Netflix Titles Added Over Time',
    markers=True
)

# 3️⃣ Build an Interactive Dashboard Using Dash
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Netflix Data Analysis Dashboard", style={'textAlign': 'center'}),

    # Genre Bar Chart
    dcc.Graph(id='genre-chart', figure=fig_genre),

    # Time-Series Trend
    dcc.Graph(id='trend-chart', figure=fig_trend),

    # Dropdown to Filter Data
    html.Label("Select Type (Movie/TV Show):"),
    dcc.Dropdown(
        id='type-dropdown',
        options=[
            {'label': 'Movies', 'value': 'Movie'},
            {'label': 'TV Shows', 'value': 'TV Show'}
        ],
        value='Movie',
        clearable=False
    ),

    # Filtered Scatter Plot
    dcc.Graph(id='scatter-plot')
])

# Callback to update scatter plot based on selected type
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('type-dropdown', 'value')
)
def update_scatter(selected_type):
    filtered_data = data[data['type'] == selected_type]
    fig = px.scatter(
        filtered_data,
        x='release_year',
        y='Duration_Minutes',
        color='rating',
        title=f'Duration of {selected_type}s by Release Year',
        labels={'Duration_Minutes': 'Duration (Minutes)', 'release_year': 'Release Year'}
    )
    return fig

# Run the dashboard
if __name__ == '__main__':
    app.run(debug=True)
